# Deep Agents

[deepagents](https://docs.langchain.com/oss/python/deepagents/overview) 可以看作是 LangChain 团队出品的 DeepResearch。本节通过查询一个近期新闻，检验 Deep Agents 是否有主动调用搜索、深度洞察的能力。

> 这里只是简单 quickstart 一下，更多信息大家去官网看吧！

In [1]:
# uv add deepagents openai

**1）加载模型**

In [2]:
import os

from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from deepagents import create_deep_agent
from openai import OpenAI

# 加载模型配置
_ = load_dotenv()

# DashScope 兼容 OpenAI Responses API
dashscope_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 加载模型（深度研究任务步数多，关闭思考以控制延迟）
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    model="qwen3.8-flash",
    extra_body={"enable_thinking": False},
)

**2）创建搜索工具**

我们使用上一节创建的 `dashscope_search` 搜索工具。

In [3]:
@tool
def dashscope_search(query: str) -> str:
    """使用夸克搜索 API 搜索互联网信息"""
    response = dashscope_client.responses.create(
        model="qwen3.8-flash", input=query, tools=[{"type": "web_search"}]
    )
    return response.output_text

**3）创建深度代理**

In [4]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """你是一名资深研究员。你的工作是进行全面深入的研究，并撰写一份精炼的报告。

你可以使用互联网搜索工具作为获取信息的主要方式。

## `dashscope_search`

使用该工具对指定查询进行互联网搜索。
"""


@tool
def get_today_date() -> str:
    """获取今天的日期"""
    return datetime.now().strftime("%Y-%m-%d")


agent = create_deep_agent(
    model=llm,
    tools=[dashscope_search, get_today_date],
    system_prompt=research_instructions,
)

**4）运行 Agent**

我们的问题是：

```
新上任的玻利维亚总统是谁？请介绍一下这位总统。
```

因为近期（2025 年 11 月 8 日）玻利维亚总统 `罗德里戈·帕斯·佩雷拉` 刚刚上任，这条信息肯定不在预训练数据里。可以用这个问题验证 Deep Agents 是否真的联网了。

In [5]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "新上任的玻利维亚总统是谁？请介绍一下这位总统。",
            }
        ]
    }
)

# 最终回复
print(result["messages"][-1].content)

# 玻利维亚现任总统：罗德里戈·帕斯（Rodrigo Paz Pereira）

玻利维亚的新任总统是**罗德里戈·帕斯·佩雷拉**，他于 **2025年11月8日** 宣誓就职，成为该国第68任总统。他的上台终结了左翼"争取社会主义运动"（MAS）长达约20年的执政，被视为拉美政治格局右转的重要标志性事件。以下是详细介绍：

## 一、个人背景
- **出生**：1967年9月22日，生于西班牙圣地亚哥德孔波斯特拉（当时其父正流亡欧洲）。
- **政治世家出身**：其父是玻利维亚前总统**海梅·帕斯·萨莫拉**（1989—1993年在任）。
- **学历**：毕业于美国大学（American University，华盛顿），获政治管理硕士学位。
- **党派**：中间偏右的**基督教民主党**。
- **从政经历**：曾任众议员、塔里哈市市长（2015—2020）、塔里哈省参议员（2020年起）。

## 二、2025年当选过程
- **首轮投票**（2025年8月17日）：帕斯得票约 **32.1%**，领先前总统基罗加（约26.7%）。
- **决胜轮**（2025年10月19日）：他以约 **54.5%** 的得票率胜出。
- **就职**：2025年11月8日在拉巴斯的多民族立法议会宣誓，任期5年，时年58岁。

## 三、执政理念："全民资本主义"
帕斯的核心口号是**"全民资本主义"（capitalism for all）**，主要政策包括：
- 削减长期拖累财政的**燃料补贴**；
- 减税、强调财政纪律、推动权力下放；
- 修改投资法，开放**锂等战略资源**领域以吸引外资；
- 重返国际资本市场、放松外汇管制；
- 外交上明显**向美国和欧洲靠拢**，修复此前紧张的对美关系。

## 四、执政首年的成效与挑战
**成效**：与国际货币基金组织（IMF）达成约19亿美元融资协议；成功发行主权债券重新进入国际市场；让玻利维亚诺汇率浮动，缓解了美元荒。

**挑战**：他接手的是玻利维亚"40年来最严重的经济危机"——高通胀、燃料和美元短缺；取消燃油补贴导致油价大幅上涨，引发工会和左翼势力的抗议与封锁；国会席位高度碎片化，改革推进阻力较大。

总体而言，帕斯是一位出身政治世家、主打市场化改革的中右翼技术官僚型总统。他正试图把玻利维亚拉出经济深渊，但执政

In [6]:
# 思考过程
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

新上任的玻利维亚总统是谁？请介绍一下这位总统。
================================== Ai Message ==================================
Tool Calls:
  get_today_date (call_3d2c0b16d029499697fdf2d0)
 Call ID: call_3d2c0b16d029499697fdf2d0
  Args:
  dashscope_search (call_e2d606266b2c49489159987f)
 Call ID: call_e2d606266b2c49489159987f
  Args:
    query: 玻利维亚 新总统 罗德里戈·帕斯
================================= Tool Message =================================
Name: get_today_date

2026-11-20
================================= Tool Message =================================
Name: dashscope_search

# 玻利维亚总统 罗德里戈·帕斯（Rodrigo Paz Pereira）

罗德里戈·帕斯·佩雷拉是玻利维亚现任（第68任）总统，他的上台标志着该国延续了约20年的左翼"争取社会主义运动"（MAS）一党长期执政的终结，是玻利维亚乃至整个拉美政治格局转向的一个重要节点。截至2026年11月，他已执政约一年。

## 一、个人背景

- **出生**：1967年9月22日，生于西班牙圣地亚哥德孔波斯特拉（当时其父流亡欧洲）。
- **政治世家**：其父是玻利维亚前总统**海梅·帕斯·萨莫拉**（Jaime Paz Zamora，1989—1993年在任）。2025年美国使馆归还一件家族遗物十字架时，这段渊源还一度成为新闻话题。
- **学历**：美国大学（American University）政治管理硕士。